## Imports

In [1]:
from IPython.display import display

import numpy as np
from matplotlib import pyplot as plt
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import PredefinedSplit, GridSearchCV

from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess

from tensordict import TensorDict
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

import xgboost as xgb

import copy
from datetime import timedelta, date
import math
import os
import random

from utils import *

c:\Users\Moral\OneDrive\Documents\Raul Castillo\Programming Fun\WeatherPrediction\notebooks\utils.py:22: DtypeWarning: Columns (7: ACMH_ATTRIBUTES, 9: ACSH_ATTRIBUTES, 13: FMTM_ATTRIBUTES, 19: PSUN_ATTRIBUTES, 25: TAVG_ATTRIBUTES, 31: TSUN_ATTRIBUTES, 33: WDF1_ATTRIBUTES, 39: WDFG_ATTRIBUTES, 41: WESD_ATTRIBUTES, 43: WSF1_ATTRIBUTES, 49: WSFG_ATTRIBUTES, 57: WT04_ATTRIBUTES, 59: WT05_ATTRIBUTES, 61: WT06_ATTRIBUTES, 63: WT07_ATTRIBUTES, 67: WT09_ATTRIBUTES, 69: WT10_ATTRIBUTES, 71: WT11_ATTRIBUTES, 73: WT13_ATTRIBUTES, 75: WT14_ATTRIBUTES, 77: WT15_ATTRIBUTES, 79: WT16_ATTRIBUTES, 81: WT18_ATTRIBUTES, 83: WT21_ATTRIBUTES, 85: WV03_ATTRIBUTES) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_ghcnd = pd.read_csv(r'../data/raw/GHCND_1980_to_2026.csv')


In [2]:
FIGURES_PATH = r'./figures'
os.makedirs(FIGURES_PATH, exist_ok=True)

In [3]:
SEED = 0

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Makes many operations deterministic
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

## Data Processing

In [4]:
raw_ghcnd = pd.read_csv(r'../data/raw/GHCND_1980_to_2026.csv')
raw_ghcnd["DATE"] = pd.to_datetime(raw_ghcnd["DATE"])

C:\Users\Moral\AppData\Local\Temp\ipykernel_7720\781470718.py:1: DtypeWarning: Columns (7: ACMH_ATTRIBUTES, 9: ACSH_ATTRIBUTES, 13: FMTM_ATTRIBUTES, 19: PSUN_ATTRIBUTES, 25: TAVG_ATTRIBUTES, 31: TSUN_ATTRIBUTES, 33: WDF1_ATTRIBUTES, 39: WDFG_ATTRIBUTES, 41: WESD_ATTRIBUTES, 43: WSF1_ATTRIBUTES, 49: WSFG_ATTRIBUTES, 57: WT04_ATTRIBUTES, 59: WT05_ATTRIBUTES, 61: WT06_ATTRIBUTES, 63: WT07_ATTRIBUTES, 67: WT09_ATTRIBUTES, 69: WT10_ATTRIBUTES, 71: WT11_ATTRIBUTES, 73: WT13_ATTRIBUTES, 75: WT14_ATTRIBUTES, 77: WT15_ATTRIBUTES, 79: WT16_ATTRIBUTES, 81: WT18_ATTRIBUTES, 83: WT21_ATTRIBUTES, 85: WV03_ATTRIBUTES) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_ghcnd = pd.read_csv(r'../data/raw/GHCND_1980_to_2026.csv')


## Baseline Models

### Persistence

In [ ]:
persistence_train_df, persistence_val_df, persistence_test_df = persistence_ghcnd_dataset(raw_ghcnd)
persistence_model = PersistenceBaseline()

#### Persistence Metrics

In [ ]:
persistence_train_output = persistence_model.predict(persistence_train_df)
persistence_train_metrics = {
    "MAE":  mean_absolute_error(persistence_train_df[["TMAX", "TMIN"]].values, persistence_train_output.values, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(persistence_train_df[["TMAX", "TMIN"]].values, persistence_train_output.values, multioutput="raw_values"),
    "ME":   mean_error(persistence_train_df[["TMAX", "TMIN"]].values, persistence_train_output.values)
}

persistence_val_output = persistence_model.predict(persistence_val_df)
persistence_val_metrics = {
    "MAE":  mean_absolute_error(persistence_val_df[["TMAX", "TMIN"]].values, persistence_val_output.values, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(persistence_val_df[["TMAX", "TMIN"]].values, persistence_val_output.values, multioutput="raw_values"),
    "ME":   mean_error(persistence_val_df[["TMAX", "TMIN"]].values, persistence_val_output.values)
}

persistence_test_output = persistence_model.predict(persistence_test_df)
persistence_test_metrics = {
    "MAE":  mean_absolute_error(persistence_test_df[["TMAX", "TMIN"]].values, persistence_test_output.values, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(persistence_test_df[["TMAX", "TMIN"]].values, persistence_test_output.values, multioutput="raw_values"),
    "ME":   mean_error(persistence_test_df[["TMAX", "TMIN"]].values, persistence_test_output.values)
}

In [ ]:
persistence_train_performance_df = persistence_train_output.join(persistence_train_df[["TMAX", "TMIN"]])
persistence_val_performance_df = persistence_val_output.join(persistence_val_df[["TMAX", "TMIN"]])
persistence_test_performance_df = persistence_test_output.join(persistence_test_df[["TMAX", "TMIN"]])

In [ ]:
print("Train Metrics: (TMAX/TMIN)")
print(f"- MAE: {persistence_train_metrics["MAE"]}")
print(f"- RMSE: {persistence_train_metrics["RMSE"]}")
print(f"- ME: {persistence_train_metrics["ME"]}\n")

print("Validation Metrics: (TMAX/TMIN)")
print(f"- MAE: {persistence_val_metrics["MAE"]}")
print(f"- RMSE: {persistence_val_metrics["RMSE"]}")
print(f"- ME: {persistence_val_metrics["ME"]}\n")

print("Test Metrics: (TMAX/TMIN)")
print(f"- MAE: {persistence_test_metrics["MAE"]}")
print(f"- RMSE: {persistence_test_metrics["RMSE"]}")
print(f"- ME: {persistence_test_metrics["ME"]}\n")


### Climatological

In [ ]:
climatological_train_df, climatological_val_df, climatological_test_df = climatological_ghcnd_dataset(raw_ghcnd) # For Climatology as above removes the first 6 days of 1980
climatological_model = ClimatologicalBaseline(climatological_train_df)

#### Climatological Metrics

In [ ]:
climatological_train_output = climatological_model.predict(climatological_train_df)
climatological_train_metrics = {
    "MAE":  mean_absolute_error(climatological_train_df[["TMAX", "TMIN"]].values, climatological_train_output.values, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(climatological_train_df[["TMAX", "TMIN"]].values, climatological_train_output.values, multioutput="raw_values"),
    "ME":   mean_error(climatological_train_df[["TMAX", "TMIN"]].values, climatological_train_output.values)
}

climatological_val_output = climatological_model.predict(climatological_val_df)
climatological_val_metrics = {
    "MAE":  mean_absolute_error(climatological_val_df[["TMAX", "TMIN"]].values, climatological_val_output.values, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(climatological_val_df[["TMAX", "TMIN"]].values, climatological_val_output.values, multioutput="raw_values"),
    "ME":   mean_error(climatological_val_df[["TMAX", "TMIN"]].values, climatological_val_output.values)
}

climatological_test_output = climatological_model.predict(climatological_test_df)
climatological_test_metrics = {
    "MAE":  mean_absolute_error(climatological_test_df[["TMAX", "TMIN"]].values, climatological_test_output.values, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(climatological_test_df[["TMAX", "TMIN"]].values, climatological_test_output.values, multioutput="raw_values"),
    "ME":   mean_error(climatological_test_df[["TMAX", "TMIN"]].values, climatological_test_output.values)
}

In [ ]:
climatological_train_performance_df = climatological_train_output.join(climatological_train_df[["TMAX", "TMIN"]])
climatological_val_performance_df = climatological_val_output.join(climatological_val_df[["TMAX", "TMIN"]])
climatological_test_performance_df = climatological_test_output.join(climatological_test_df[["TMAX", "TMIN"]])

In [ ]:
print("Train Metrics: (TMAX/TMIN)")
print(f"- MAE: {climatological_train_metrics["MAE"]}")
print(f"- RMSE: {climatological_train_metrics["RMSE"]}")
print(f"- ME: {climatological_train_metrics["ME"]}\n")

print("Validation Metrics: (TMAX/TMIN)")
print(f"- MAE: {climatological_val_metrics["MAE"]}")
print(f"- RMSE: {climatological_val_metrics["RMSE"]}")
print(f"- ME: {climatological_val_metrics["ME"]}\n")

print("Test Metrics: (TMAX/TMIN)")
print(f"- MAE: {climatological_test_metrics["MAE"]}")
print(f"- RMSE: {climatological_test_metrics["RMSE"]}")
print(f"- ME: {climatological_test_metrics["ME"]}\n")

### Comparing Climatology with Persistence
$$\text{Baseline Skill} = 1 - \frac{\text{MAE}_{persistence}}{\text{MAE}_{climatological}}$$

In [ ]:
print("Train Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(climatological_train_metrics, persistence_train_metrics)}\n")

print("Validation Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(climatological_val_metrics, persistence_val_metrics)}\n")

print("Test Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(climatological_test_metrics, persistence_test_metrics)}\n")

## ML Models

### Linear Regression

In [ ]:
linear_train_df, linear_val_df, linear_test_df = linear_ghcnd_dataset(raw_ghcnd)

In [ ]:
linear_train_X, linear_train_y = linear_train_df.drop(columns=["TMAX", "TMIN"]).values, linear_train_df[["TMAX", "TMIN"]].values
linear_val_X, linear_val_y = linear_val_df.drop(columns=["TMAX", "TMIN"]).values, linear_val_df[["TMAX", "TMIN"]].values
linear_test_X, linear_test_y = linear_test_df.drop(columns=["TMAX", "TMIN"]).values, linear_test_df[["TMAX", "TMIN"]].values

In [ ]:
linear_model = LinearRegression()
linear_model.fit(linear_train_X, linear_train_y)

In [ ]:
linear_train_normalized_output = linear_model.predict(linear_train_X)
linear_val_normalized_output = linear_model.predict(linear_val_X)
linear_test_normalized_output = linear_model.predict(linear_test_X)

linear_train_unnormalized_output = linear_train_normalized_output*TRAIN_STD.values + TRAIN_MEAN.values
linear_val_unnormalized_output = linear_val_normalized_output*TRAIN_STD.values + TRAIN_MEAN.values
linear_test_unnormalized_output = linear_test_normalized_output*TRAIN_STD.values + TRAIN_MEAN.values

linear_train_unnormalized_y = linear_train_y*TRAIN_STD.values + TRAIN_MEAN.values
linear_val_unnormalized_y = linear_val_y*TRAIN_STD.values + TRAIN_MEAN.values
linear_test_unnormalized_y = linear_test_y*TRAIN_STD.values + TRAIN_MEAN.values

In [ ]:
linear_train_performance_df = pd.DataFrame(linear_train_unnormalized_output, index=linear_train_df.index, columns=["predicted_TMAX", "predicted_TMIN"]).join(pd.DataFrame(linear_train_unnormalized_y, index=linear_train_df.index, columns=["TMAX", "TMIN"]))
linear_val_performance_df = pd.DataFrame(linear_val_unnormalized_output, index=linear_val_df.index, columns=["predicted_TMAX", "predicted_TMIN"]).join(pd.DataFrame(linear_val_unnormalized_y, index=linear_val_df.index, columns=["TMAX", "TMIN"]))
linear_test_performance_df = pd.DataFrame(linear_test_unnormalized_output, index=linear_test_df.index, columns=["predicted_TMAX", "predicted_TMIN"]).join(pd.DataFrame(linear_test_unnormalized_y, index=linear_test_df.index, columns=["TMAX", "TMIN"]))

#### Linear Metrics

In [ ]:
linear_train_metrics = {
    "MAE":  mean_absolute_error(linear_train_unnormalized_output, linear_train_unnormalized_y, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(linear_train_unnormalized_output, linear_train_unnormalized_y, multioutput="raw_values"),
    "ME":   mean_error(linear_train_unnormalized_output, linear_train_unnormalized_y)
}
linear_val_metrics = {
    "MAE":  mean_absolute_error(linear_val_unnormalized_output, linear_val_unnormalized_y, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(linear_val_unnormalized_output, linear_val_unnormalized_y, multioutput="raw_values"),
    "ME":   mean_error(linear_val_unnormalized_output, linear_val_unnormalized_y)
}
linear_test_metrics = {
    "MAE":  mean_absolute_error(linear_test_unnormalized_output, linear_test_unnormalized_y, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(linear_test_unnormalized_output, linear_test_unnormalized_y, multioutput="raw_values"),
    "ME":   mean_error(linear_test_unnormalized_output, linear_test_unnormalized_y)
}

In [ ]:
print("Train Metrics: (TMAX/TMIN)")
print(f"- MAE: {linear_train_metrics["MAE"]}")
print(f"- RMSE: {linear_train_metrics["RMSE"]}")
print(f"- ME: {linear_train_metrics["ME"]}\n")

print("Validation Metrics: (TMAX/TMIN)")
print(f"- MAE: {linear_val_metrics["MAE"]}")
print(f"- RMSE: {linear_val_metrics["RMSE"]}")
print(f"- ME: {linear_val_metrics["ME"]}\n")

print("Test Metrics: (TMAX/TMIN)")
print(f"- MAE: {linear_test_metrics["MAE"]}")
print(f"- RMSE: {linear_test_metrics["RMSE"]}")
print(f"- ME: {linear_test_metrics["ME"]}\n")

#### Comparing Linear with Climatology

In [ ]:
print("Train Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(climatological_train_metrics, linear_train_metrics)}\n")

print("Validation Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(climatological_val_metrics, linear_val_metrics)}\n")

print("Test Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(climatological_test_metrics, linear_test_metrics)}\n")

#### Comparing Linear with Persistence

In [ ]:
print("Train Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(persistence_train_metrics, linear_train_metrics)}\n")

print("Validation Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(persistence_val_metrics, linear_val_metrics)}\n")

print("Test Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(persistence_test_metrics, linear_test_metrics)}\n")

### XGBoost

In [ ]:
xgboost_train_df, xgboost_val_df, xgboost_test_df = xgboost_ghcnd_dataset(raw_ghcnd)

In [ ]:
xgboost_train_X, xgboost_train_y = xgboost_train_df.drop(columns=["TMAX", "TMIN"]), xgboost_train_df[["TMAX", "TMIN"]]
xgboost_val_X, xgboost_val_y = xgboost_val_df.drop(columns=["TMAX", "TMIN"]), xgboost_val_df[["TMAX", "TMIN"]]
xgboost_test_X, xgboost_test_y = xgboost_test_df.drop(columns=["TMAX", "TMIN"]), xgboost_test_df[["TMAX", "TMIN"]]

In [ ]:
xgboost_train_val_X = np.concatenate((xgboost_train_X, xgboost_val_X), axis=0)
xgboost_train_val_y = np.concatenate((xgboost_train_y, xgboost_val_y), axis=0)

test_fold = [-1]*len(xgboost_train_X) + [0]*len(xgboost_val_X)
ps = PredefinedSplit(test_fold=test_fold)

grid_search = GridSearchCV(
    estimator=xgb.XGBRegressor(random_state=SEED),
    param_grid={
        "n_estimators": [50, 100, 200],
        "learning_rate": [0.01, 0.1],
        "max_depth": [3, 6, 12],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "gamma": [0, 0.1]
    },
    scoring="neg_mean_squared_error",
    cv=ps,
    refit=False,
    n_jobs=-1
)

grid_search.fit(xgboost_train_val_X, xgboost_train_val_y)

In [ ]:
grid_search.best_params_

In [ ]:
xgboost_model = xgb.XGBRegressor(**grid_search.best_params_)
xgboost_model.fit(xgboost_train_X, xgboost_train_y)

In [ ]:
xgboost_train_normalized_output = xgboost_model.predict(xgboost_train_X)
xgboost_val_normalized_output = xgboost_model.predict(xgboost_val_X)
xgboost_test_normalized_output = xgboost_model.predict(xgboost_test_X)

xgboost_train_unnormalized_output = xgboost_train_normalized_output*TRAIN_STD.values + TRAIN_MEAN.values
xgboost_val_unnormalized_output = xgboost_val_normalized_output*TRAIN_STD.values + TRAIN_MEAN.values
xgboost_test_unnormalized_output = xgboost_test_normalized_output*TRAIN_STD.values + TRAIN_MEAN.values

xgboost_train_unnormalized_y = xgboost_train_y.values*TRAIN_STD.values + TRAIN_MEAN.values
xgboost_val_unnormalized_y = xgboost_val_y.values*TRAIN_STD.values + TRAIN_MEAN.values
xgboost_test_unnormalized_y = xgboost_test_y.values*TRAIN_STD.values + TRAIN_MEAN.values

In [ ]:
xgboost_train_performance_df = pd.DataFrame(xgboost_train_unnormalized_output, index=xgboost_train_df.index, columns=["predicted_TMAX", "predicted_TMIN"]).join(pd.DataFrame(xgboost_train_unnormalized_y, index=xgboost_train_df.index, columns=["TMAX", "TMIN"]))
xgboost_val_performance_df = pd.DataFrame(xgboost_val_unnormalized_output, index=xgboost_val_df.index, columns=["predicted_TMAX", "predicted_TMIN"]).join(pd.DataFrame(xgboost_val_unnormalized_y, index=xgboost_val_df.index, columns=["TMAX", "TMIN"]))
xgboost_test_performance_df = pd.DataFrame(xgboost_test_unnormalized_output, index=xgboost_test_df.index, columns=["predicted_TMAX", "predicted_TMIN"]).join(pd.DataFrame(xgboost_test_unnormalized_y, index=xgboost_test_df.index, columns=["TMAX", "TMIN"]))

#### XGBoost Metrics

In [ ]:
xgboost_train_metrics = {
    "MAE":  mean_absolute_error(xgboost_train_unnormalized_output, xgboost_train_unnormalized_y, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(xgboost_train_unnormalized_output, xgboost_train_unnormalized_y, multioutput="raw_values"),
    "ME":   mean_error(xgboost_train_unnormalized_output, xgboost_train_unnormalized_y)
}
xgboost_val_metrics = {
    "MAE":  mean_absolute_error(xgboost_val_unnormalized_output, xgboost_val_unnormalized_y, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(xgboost_val_unnormalized_output, xgboost_val_unnormalized_y, multioutput="raw_values"),
    "ME":   mean_error(xgboost_val_unnormalized_output, xgboost_val_unnormalized_y)
}
xgboost_test_metrics = {
    "MAE":  mean_absolute_error(xgboost_test_unnormalized_output, xgboost_test_unnormalized_y, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(xgboost_test_unnormalized_output, xgboost_test_unnormalized_y, multioutput="raw_values"),
    "ME":   mean_error(xgboost_test_unnormalized_output, xgboost_test_unnormalized_y)
}

In [ ]:
print("Train Metrics: (TMAX/TMIN)")
print(f"- MAE: {xgboost_train_metrics["MAE"]}")
print(f"- RMSE: {xgboost_train_metrics["RMSE"]}")
print(f"- ME: {xgboost_train_metrics["ME"]}\n")

print("Validation Metrics: (TMAX/TMIN)")
print(f"- MAE: {xgboost_val_metrics["MAE"]}")
print(f"- RMSE: {xgboost_val_metrics["RMSE"]}")
print(f"- ME: {xgboost_val_metrics["ME"]}\n")

print("Test Metrics: (TMAX/TMIN)")
print(f"- MAE: {xgboost_test_metrics["MAE"]}")
print(f"- RMSE: {xgboost_test_metrics["RMSE"]}")
print(f"- ME: {xgboost_test_metrics["ME"]}\n")

#### Comparing XGBoost with Climatological

In [ ]:
print("Train Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(climatological_train_metrics, xgboost_train_metrics)}\n")

print("Validation Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(climatological_val_metrics, xgboost_val_metrics)}\n")

print("Test Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(climatological_test_metrics, xgboost_test_metrics)}\n")

#### Comparing XGBoost with Persistence Baseline

In [ ]:
print("Train Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(persistence_train_metrics, xgboost_train_metrics)}\n")

print("Validation Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(persistence_val_metrics, xgboost_val_metrics)}\n")

print("Test Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(persistence_test_metrics, xgboost_test_metrics)}\n")

#### Decision Tree Design & Feature Importance Dict

In [ ]:
TREE_PLOT_DIR = 'tree_plots'
os.makedirs(os.path.join(FIGURES_PATH, TREE_PLOT_DIR), exist_ok=True)

In [ ]:
num_trees = len(xgboost_model.get_booster().get_dump())
for tree_idx in range(0, num_trees, 50):
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(10,10))
    xgb.plot_tree(xgboost_model, ax=ax, tree_idx=tree_idx)
    ax.set_title(f'XGBoost Tree (idx={tree_idx})', fontsize=10)
    ax.grid(False)
    fig.savefig(os.path.join(FIGURES_PATH, TREE_PLOT_DIR, f'xgboost_tree_idx={tree_idx}.png'))

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(10,7))
xgb.plot_importance(xgboost_model, ax=ax)
ax.set_title('XGBoost Feature Importance')
ax.grid(False)
fig.savefig(os.path.join(FIGURES_PATH, TREE_PLOT_DIR, r'xgboost_feature_plot.png'))
plt.show()

### RNN

In [5]:
rnn_train_df = raw_ghcnd[(1980 <= raw_ghcnd.DATE.map(lambda x: x.year)) & (raw_ghcnd.DATE.map(lambda x: x.year) <= 2010)]
rnn_val_df = raw_ghcnd[raw_ghcnd.DATE.between('2010-12-26', '2018-01-01', inclusive="left")] # The start at 6 days before new years day enables the collections of lag features from 6 days back
rnn_test_df = raw_ghcnd[raw_ghcnd.DATE.between('2017-12-26', '2026-01-01', inclusive="left")] # The start at 6 days before new years day enables the collections of lag features from 6 days back

In [6]:
BATCH_SIZE = 64
EPOCHS = 100

In [7]:
rnn_train_dataset = RNN_GHCND_Dataset(rnn_train_df)
rnn_val_dataset = RNN_GHCND_Dataset(rnn_val_df)
rnn_test_dataset = RNN_GHCND_Dataset(rnn_test_df)

rnn_train_dataloader = DataLoader(rnn_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
rnn_val_dataloader = DataLoader(rnn_val_dataset, batch_size=BATCH_SIZE, shuffle=True)
rnn_test_dataloader = DataLoader(rnn_test_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [9]:
next(iter(rnn_train_dataloader))

TensorDict(batch_size=[64], fields={
    fourier: f32[2],
    lag_features: f32[6,4],
    target: f32[2],
    target_date: i32[]})

In [10]:
device = torch.device(torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu")
loss_fn = nn.MSELoss()

best_state_dict = None
best_hyperparameters = None
best_val_metrics = None

for NN_HIDDEN_SIZE in [16, 32, 64, 128]:
    for RNN_HIDDEN_SIZE in [16, 32, 64, 128]:
        for LEARNING_RATE in [0.001, 0.01, 0.1]:
            print(f"Config: (NN_HIDDEN: {NN_HIDDEN_SIZE} | RNN_HIDDEN: {RNN_HIDDEN_SIZE} | LR: {LEARNING_RATE})")
            
            rnn_model = WeatherRNN(RNN_HIDDEN_SIZE, NN_HIDDEN_SIZE).to(device)
            optimizer = optim.Adam(rnn_model.parameters(), lr=LEARNING_RATE)

            local_best_val_metrics = None
            local_best_state_dict = None

            local_end = 0
            current_trend = 0
            for epoch_index in range(EPOCHS):
                rnn_train_metrics = train_one_epoch(rnn_model, loss_fn, device, rnn_train_dataloader, optimizer)
                rnn_val_metrics, _ = evaluate_model(rnn_model, loss_fn, device, rnn_val_dataloader)

                if (local_best_val_metrics is None or rnn_val_metrics["RMSE"].mean() < local_best_val_metrics["RMSE"].mean()):
                    local_best_val_metrics = rnn_val_metrics
                    local_best_state_dict = copy.deepcopy(rnn_model.state_dict())
                    local_end = epoch_index + 1
                    current_trend = 0
                else:
                    current_trend += 1

                if (current_trend >= 5):
                    break

            print(f"- Final Metrics:")
            print(f"\t- MAE: {local_best_val_metrics["MAE"]}")
            print(f"\t- RMSE: {local_best_val_metrics["RMSE"]}")
            print(f"\t- ME: {local_best_val_metrics["ME"]}")
            print(f"\t- Loss: {local_best_val_metrics["total_loss"]}")
            print(f"\t- Last Epoch: {local_end}")

            if (best_val_metrics is None or local_best_val_metrics["RMSE"].mean() < best_val_metrics["RMSE"].mean()):
                best_state_dict = local_best_state_dict
                best_val_metrics = local_best_val_metrics
                best_hyperparameters = {
                    "nn_hidden_size": NN_HIDDEN_SIZE,
                    "rnn_hidden_size": RNN_HIDDEN_SIZE,
                    "lr": LEARNING_RATE,
                    "epochs": local_end
                }

best_rnn_model = WeatherRNN(RNN_HIDDEN=best_hyperparameters["rnn_hidden_size"], NN_HIDDEN=best_hyperparameters['nn_hidden_size'])
best_rnn_model.load_state_dict(best_state_dict)
best_rnn_model.to(device)
best_rnn_model.eval()

rnn_train_metrics, rnn_train_performance_df = evaluate_model(best_rnn_model, loss_fn, device, rnn_train_dataloader)
rnn_val_metrics, rnn_val_performance_df = evaluate_model(best_rnn_model, loss_fn, device, rnn_val_dataloader)
rnn_test_metrics, rnn_test_performance_df = evaluate_model(best_rnn_model, loss_fn, device, rnn_test_dataloader)

Config: (NN_HIDDEN: 16 | RNN_HIDDEN: 16 | LR: 0.001)
- Final Metrics:
	- MAE: [2.7003624  2.50235249]
	- RMSE: [3.72033332 3.51342748]
	- ME: [-0.08538344 -0.29079106]
	- Loss: 0.13061672653660533
	- Last Epoch: 18
Config: (NN_HIDDEN: 16 | RNN_HIDDEN: 16 | LR: 0.01)
- Final Metrics:
	- MAE: [2.707781   2.59571994]
	- RMSE: [3.71418237 3.53799223]
	- ME: [-0.05719522 -0.52463882]
	- Loss: 0.1310478149766469
	- Last Epoch: 10
Config: (NN_HIDDEN: 16 | RNN_HIDDEN: 16 | LR: 0.1)
- Final Metrics:
	- MAE: [2.81647535 2.75997517]
	- RMSE: [4.00054576 3.70950774]
	- ME: [ 0.73544743 -0.02785582]
	- Loss: 0.14896465070461173
	- Last Epoch: 4
Config: (NN_HIDDEN: 16 | RNN_HIDDEN: 32 | LR: 0.001)
- Final Metrics:
	- MAE: [2.72229409 2.5303228 ]
	- RMSE: [3.72482817 3.52102818]
	- ME: [-0.21488347 -0.4115495 ]
	- Loss: 0.1310277504057543
	- Last Epoch: 18
Config: (NN_HIDDEN: 16 | RNN_HIDDEN: 32 | LR: 0.01)
- Final Metrics:
	- MAE: [2.71079591 2.60306743]
	- RMSE: [3.70146394 3.55645763]
	- ME: [-0.2

#### RNN Metrics

In [14]:
best_val_metrics

{'MAE': array([2.67140154, 2.50763217]),
 'RMSE': array([3.71502911, 3.50156075]),
 'ME': array([ 0.06518484, -0.21980092]),
 'total_loss': 0.13005076821399564}

In [15]:
best_hyperparameters

{'nn_hidden_size': 32, 'rnn_hidden_size': 128, 'lr': 0.001, 'epochs': 10}

In [ ]:
print("Train Metrics: (TMAX/TMIN)")
print(f"- MAE: {rnn_train_metrics["MAE"]}")
print(f"- RMSE: {rnn_train_metrics["RMSE"]}")
print(f"- ME: {rnn_train_metrics["ME"]}\n")

print("Validation Metrics: (TMAX/TMIN)")
print(f"- MAE: {rnn_val_metrics["MAE"]}")
print(f"- RMSE: {rnn_val_metrics["RMSE"]}")
print(f"- ME: {rnn_val_metrics["ME"]}\n")

print("Test Metrics: (TMAX/TMIN)")
print(f"- MAE: {rnn_test_metrics["MAE"]}")
print(f"- RMSE: {rnn_test_metrics["RMSE"]}")
print(f"- ME: {rnn_test_metrics["ME"]}\n")

#### Comparing RNN with Climatological

In [ ]:
print("Train Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(climatological_train_metrics, xgboost_train_metrics)}\n")

print("Validation Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(climatological_val_metrics, xgboost_val_metrics)}\n")

print("Test Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(climatological_test_metrics, xgboost_test_metrics)}\n")

#### Comparing RNN with Persistence

In [ ]:
print("Train Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(persistence_train_metrics, xgboost_train_metrics)}\n")

print("Validation Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(persistence_val_metrics, xgboost_val_metrics)}\n")

print("Test Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(persistence_test_metrics, xgboost_test_metrics)}\n")

# Final Performance Summary Plots

## Train, Val, Test Set Predictions Overtime

## Metrics

In [ ]:
METRIC_PLOT_DIR = 'metric_plots'
os.makedirs(os.path.join(FIGURES_PATH, METRIC_PLOT_DIR), exist_ok=True)

In [ ]:
def plot_model_metrics(metric="MAE"):
    # Group each model's train, validation, and test metrics in split order.
    model_metrics = {
        "Persistence": (
            persistence_train_metrics,
            persistence_val_metrics,
            persistence_test_metrics,
        ),
        "Climatological": (
            climatological_train_metrics,
            climatological_val_metrics,
            climatological_test_metrics,
        ),
        "Linear": (
            linear_train_metrics,
            linear_val_metrics,
            linear_test_metrics,
        ),
        "XGBoost": (
            xgboost_train_metrics,
            xgboost_val_metrics,
            xgboost_test_metrics,
        ),
        "LSTM": (
            rnn_train_metrics,
            rnn_val_metrics,
            rnn_test_metrics,
        ),
    }

    # Validate that only supported error metrics are requested.
    if metric not in {"MAE", "RMSE", "ME"}:
        raise ValueError("metric must be 'MAE', 'RMSE', or 'ME'")

    # Define ordering and labels used consistently by the plot.
    split_names = ["Train", "Validation", "Test"]
    target_names = ["TMAX", "TMIN"]
    model_names = list(model_metrics)

    # Create one side-by-side chart for TMAX and TMIN.
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(15, 6),
        sharex=True,
        sharey=True,
        constrained_layout=True,
    )

    # Control bar thickness and the offsets for grouping split bars.
    bar_height = 0.18
    split_spacing = 0.20
    model_positions = np.arange(len(model_names))

    # Draw one target variable in each subplot.
    for ax, target_name in zip(axes, target_names):
        bars_by_split = []

        # Plot Train, Validation, and Test bars beside each model position.
        for split_index, split_name in enumerate(split_names):
            values = [
                model_metrics[model_name][split_index][metric][
                    target_names.index(target_name)
                ]
                for model_name in model_names
            ]

            positions = model_positions + (split_index - 1) * split_spacing
            bars = ax.barh(
                positions,
                values,
                height=bar_height,
                label=split_name,
            )
            bars_by_split.append(bars)

            # Add the numeric value at the end of each horizontal bar.
            ax.bar_label(
                bars,
                labels=[
                    f"{value:.2f}" if abs(value) >= 0.01 else f"{value:.2e}"
                    for value in values
                ],
                padding=3,
                fontsize=8,
            )

        # Format axes, add a zero reference line, and improve readability.
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_title(f"{target_name} — {metric} (°F)")
        ax.set_xlabel(metric)
        ax.set_yticks(model_positions)
        ax.set_yticklabels(model_names)
        ax.set_yinverted(True)
        ax.grid(axis="x", linestyle="--", alpha=0.3)
        ax.margins(x=0.1, y=0.01)

    # Add shared labels and legend, then save and display the figure.
    axes[0].set_ylabel("Model")
    axes[1].legend(title="Split")
    fig.suptitle(f"Train, Validation, and Test {metric} by Model")

    figure_path = os.path.join(
        FIGURES_PATH,
        METRIC_PLOT_DIR,
        f"model_{metric}_comparison.png",
    )
    fig.savefig(figure_path, dpi=300, bbox_inches="tight")
    plt.show()

    return fig, axes


# Generate and save a comparison chart for each supported metric.
for metric_name in ["MAE", "RMSE", "ME"]:
    plot_model_metrics(metric_name)

## Baselines

In [ ]:
BASELINE_PLOT_DIR = 'baseline_plots'
os.makedirs(os.path.join(FIGURES_PATH, BASELINE_PLOT_DIR), exist_ok=True)

In [ ]:
model_metrics = {
    "Persistence": (
        persistence_train_metrics,
        persistence_val_metrics,
        persistence_test_metrics,
    ),
    "Climatological": (
        climatological_train_metrics,
        climatological_val_metrics,
        climatological_test_metrics,
    ),
    "Linear": (
        linear_train_metrics,
        linear_val_metrics,
        linear_test_metrics,
    ),
    "XGBoost": (
        xgboost_train_metrics,
        xgboost_val_metrics,
        xgboost_test_metrics,
    ),
    "LSTM": (
        rnn_train_metrics,
        rnn_val_metrics,
        rnn_test_metrics,
    ),
}

baseline_metrics = {
    "Climatological": (
        climatological_train_metrics,
        climatological_val_metrics,
        climatological_test_metrics,
    ),
    "Persistence": (
        persistence_train_metrics,
        persistence_val_metrics,
        persistence_test_metrics,
    ),
}

split_names = ["Train", "Validation", "Test"]
model_names = list(model_metrics)
target_names = ["TMAX", "TMIN"]

for baseline_name, split_baselines in baseline_metrics.items():
    fig, axes = plt.subplots(
        1, 2, figsize=(15, 6), sharex=True, sharey=True, constrained_layout=True
    )

    model_positions = np.arange(len(model_names))
    split_spacing = 0.20
    bar_height = 0.18

    for target_index, (ax, target_name) in enumerate(zip(axes, target_names)):
        for split_index, split_name in enumerate(split_names):
            baseline = split_baselines[split_index]
            skills = [
                baseline_skill(baseline, model_metrics[model_name][split_index])[target_index]
                for model_name in model_names
            ]
            positions = model_positions + (split_index - 1) * split_spacing

            bars = ax.barh(
                positions,
                skills,
                height=bar_height,
                label=split_name,
            )
            ax.bar_label(
                bars,
                labels=[f"{skill:.2%}" for skill in skills],
                padding=3,
                fontsize=8,
            )

        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_title(f"{target_name} — Skill vs. {baseline_name}")
        ax.set_xlabel("Baseline skill")
        ax.set_yticks(model_positions)
        ax.set_yticklabels(model_names)
        ax.set_yinverted(True)
        ax.grid(axis="x", linestyle="--", alpha=0.3)
        ax.margins(x=0.15, y=0.01)

    axes[0].set_ylabel("Model")
    axes[1].legend(title="Split")
    fig.suptitle(f"Train, Validation, and Test Baseline Skill vs. {baseline_name}")

    figure_path = os.path.join(
        FIGURES_PATH,
        BASELINE_PLOT_DIR,
        f"baseline_skill_{baseline_name.lower()}.png",
    )
    fig.savefig(figure_path, dpi=300, bbox_inches="tight")
    plt.show()